# 🔄 Notebook 3 — Run RAG & Capture Outputs
## RAGAS Evaluation Series · Part 3 of 4

---

## 🎯 What this notebook does

We load the evaluation questions from Notebook 2, run each one through the full RAG pipeline,  
and capture the three things RAGAS needs for every question:

```
For each question RAGAS needs:
┌─────────────────────────────────────────────────────┐
│  question   — the original question                  │
│  answer     — what the LLM generated                 │
│  contexts   — the list of chunks the retriever found │
└─────────────────────────────────────────────────────┘
```

Then we merge the RAG outputs with the ground truths from Notebook 2  
to produce the complete `ragas_input.json` that Notebook 4 will evaluate.

---

## 🔄 Data flow

```
evaluation_dataset.json   (from Notebook 2)
        │
        │  question list
        ▼
   RAG Pipeline
   (retriever + LLM)
        │
        │  question + answer + contexts
        ▼
   Merge with ground_truth
        │
        ▼
   ragas_input.json   ──→  Notebook 4 evaluates this
```

In [1]:
# ── Install (uncomment if needed) ────────────────────────────────────────────
# !pip install langchain langchain-community langchain-ollama langchain-chroma chromadb

---
## 📚 Step 0 — Imports & Setup

In [2]:
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
import json

print("✅ Imports successful")

C:\Users\sr43993\AppData\Local\Temp\ipykernel_33692\4049694121.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader
d:\RAG Practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports successful


---
## 📂 Step 1 — Load the Evaluation Dataset

Load the questions and ground truths created in Notebook 2.

In [3]:
# ── Load evaluation dataset from Notebook 2 ──────────────────────────────────
with open("evaluation_dataset.json") as f:
    evaluation_data = json.load(f)

print(f"✅ Loaded {len(evaluation_data)} evaluation entries")
print()
for i, entry in enumerate(evaluation_data, 1):
    print(f"  [{i}] {entry['question']}")

✅ Loaded 5 evaluation entries

  [1] What is the maternity leave policy?
  [2] How many days annual leave do employees get?
  [3] What is the notice period before resignation?
  [4] Can employees work from home?
  [5] What is the probation period for new employees?


---
## 🔧 Step 2 — Rebuild the RAG Pipeline

We rebuild the same pipeline from Notebook 1.  
If you already have the ChromaDB persisted, Step 3 (embedding) will load from disk instantly.

In [4]:
# ── Rebuild RAG pipeline ─────────────────────────────────────────────────────
# Step 2a: Load and chunk document
loader = TextLoader("docs/company_policy.txt", encoding="utf-8")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
print(f"✅ Document loaded — {len(chunks)} chunks")

# Step 2b: Embeddings + vector store (loads from disk if ./chroma_db exists)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"✅ Retriever ready — {vectorstore._collection.count()} vectors")

# Step 2c: LLM
llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)
print("✅ LLM ready")

✅ Document loaded — 1 chunks
✅ Retriever ready — 2 vectors
✅ LLM ready


---
## 🏃 Step 3 — Run RAG on Every Evaluation Question

For each question we:
1. Retrieve the top-3 most relevant chunks
2. Build the prompt with those chunks as context
3. Ask the LLM to answer
4. Save question + answer + contexts

> ⚠️ **contexts must be a list of strings** (not Document objects).  
> RAGAS expects `List[str]` — we extract `.page_content` from each retrieved Document.

In [5]:
# ── Run RAG on each evaluation question ──────────────────────────────────────
#
# For each question we save:
#   question:  str        — the original question
#   answer:    str        — LLM generated answer
#   contexts:  List[str]  — list of retrieved chunk texts (RAGAS expects List[str])

ragas_inputs = []

print("Running RAG on evaluation questions...\n")

for i, entry in enumerate(evaluation_data, 1):
    question = entry["question"]
    ground_truth = entry["ground_truth"]

    # ── Retrieve ──────────────────────────────────────────────────────────────
    retrieved_docs = retriever.invoke(question)

    # IMPORTANT: extract page_content strings — RAGAS needs List[str], not List[Document]
    contexts = [doc.page_content for doc in retrieved_docs]

    # ── Build context string ──────────────────────────────────────────────────
    context_str = "\n\n".join(contexts)

    # ── Prompt ────────────────────────────────────────────────────────────────
    prompt = f"""You are a helpful assistant.
Answer ONLY using the provided context.
If the answer is not present, reply: "I could not find that information in the document."

Context:
{context_str}

Question:
{question}
"""

    # ── Generate answer ───────────────────────────────────────────────────────
    response = llm.invoke(prompt)
    answer = response.content

    # ── Store result ──────────────────────────────────────────────────────────
    ragas_inputs.append({
        "question":     question,
        "answer":       answer,
        "contexts":     contexts,
        "ground_truth": ground_truth,
    })

    print(f"  [{i}/{len(evaluation_data)}] ✅ {question}")
    print(f"       Answer: {answer[:90]}...")
    print()

print(f"\n✅ All {len(ragas_inputs)} questions processed")

Running RAG on evaluation questions...

  [1/5] ✅ What is the maternity leave policy?
       Answer: Employees are eligible for 6 months maternity leave....

  [2/5] ✅ How many days annual leave do employees get?
       Answer: 24 days....

  [3/5] ✅ What is the notice period before resignation?
       Answer: The notice period before resignation is 60 days....

  [4/5] ✅ Can employees work from home?
       Answer: Yes—employees are allowed to work from home twice a week....

  [5/5] ✅ What is the probation period for new employees?
       Answer: I could not find that information in the document....


✅ All 5 questions processed


---
## 🔍 Step 4 — Inspect the Captured Data

Before saving, let's verify the structure is correct.  
RAGAS needs exactly this format.

In [6]:
# ── Inspect first entry ───────────────────────────────────────────────────────
entry = ragas_inputs[0]

print("📋 RAGAS input format (first entry):")
print()
print(f"  question     : {entry['question']}")
print(f"  answer       : {entry['answer'][:100]}...")
print(f"  ground_truth : {entry['ground_truth']}")
print(f"  contexts     : {len(entry['contexts'])} chunks")
for i, ctx in enumerate(entry['contexts'], 1):
    print(f"    [{i}] {ctx[:100]}...")

print()
print("✅ Structure correct — question / answer / contexts / ground_truth all present")

📋 RAGAS input format (first entry):

  question     : What is the maternity leave policy?
  answer       : Employees are eligible for 6 months maternity leave....
  ground_truth : Employees are eligible for 6 months maternity leave.
  contexts     : 2 chunks
    [1] Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period bef...
    [2] Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period bef...

✅ Structure correct — question / answer / contexts / ground_truth all present


---
## 💾 Step 5 — Save RAGAS Input File

In [7]:
# ── Save to ragas_input.json ──────────────────────────────────────────────────
# This is the file Notebook 4 will load and pass directly to RAGAS

with open("ragas_input.json", "w") as f:
    json.dump(ragas_inputs, f, indent=2)

print(f"💾 Saved ragas_input.json — {len(ragas_inputs)} entries")
print()
print("➡️  Next: open Notebook 4 to run RAGAS and see your scores.")

💾 Saved ragas_input.json — 5 entries

➡️  Next: open Notebook 4 to run RAGAS and see your scores.
